# Neural Feature Extraction (Whisper + XLS-R) — Kaggle GPU

Output: `tokens.csv`, `features_whisper.npz`, `features_xlsr.npz`

> Download  files at the end of the run and drop them into local `project/data/features/` directory. Then run `python scripts/normalise.py` and `python scripts/analyse.py` locally.

## 1. Setup

In [1]:
!pip install -q transformers librosa textgrid praat-parselmouth 2>&1 | tail -3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 89.9 MB/s eta 0:00:00


In [2]:
import os, glob, re, numpy as np, pandas as pd, torch, librosa
from pathlib import Path
import textgrid
import re, glob
from pathlib import Path
import pandas as pd
import textgrid
import gc
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm
from transformers import WhisperModel, WhisperFeatureExtractor





In [3]:
CORPUS_ROOT = Path("/kaggle/input/datasets/alenamuravyeva/ru-fr-interference/ru-fr_interference")
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)

WHISPER_MODEL = "openai/whisper-medium"
WHISPER_LAYERS = [4, 20]
XLSR_MODEL = "facebook/wav2vec2-large-xlsr-53"
XLSR_LAYERS = [4, 12, 20]
SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPUs:", torch.cuda.device_count())
print("corpus exists:", CORPUS_ROOT.exists())

device: cuda | GPUs: 2
corpus exists: True


## 2. Build tokens.csv (or reuse uploaded one)

In [4]:
L1_MAP = {"rus": "L1_RU", "fra": "L1_FR"}
SKIP_LABELS = {"sil", "sp", "<p:>", "", "_"}
FNAME_RE = re.compile(
    r"^(?P<spk>[a-z0-9]+)_(?P<lang>[a-z]+)_list(?P<list>\d+)_FRcorp(?P<trial>\d+)$",
    re.IGNORECASE,
)


def load_metadata(path):
    meta = pd.read_csv(path, sep=";")
    meta.columns = [c.strip().lower().replace(" ", "_") for c in meta.columns]
    meta = meta.rename(columns={"spk": "speaker"})
    meta["speaker"] = meta["speaker"].str.upper().str.strip()
    meta["gender"] = meta["gender"].str.upper().str.strip()
    return meta[["speaker", "gender"]]


def parse_filename(stem):
    m = FNAME_RE.match(stem)
    if not m or m["lang"].lower() not in L1_MAP:
        return None
    return {
        "speaker": m["spk"].upper(),
        "l1": L1_MAP[m["lang"].lower()],
        "list_id": int(m["list"]),
        "trial_id": int(m["trial"]),
    }


def load_phones_tier(tg_path):
    try:
        tg = textgrid.TextGrid.fromFile(tg_path)
    except Exception:
        return None
    return next((t for t in tg.tiers if t.name.lower() == "phones"), None)


def extract_tokens(tg_path, info):
    wav = Path(tg_path).with_suffix(".wav")
    if not wav.exists():
        return []
    tier = load_phones_tier(tg_path)
    if tier is None:
        return []

    rows, pos = [], 0
    for iv in tier.intervals:
        lab = iv.mark.strip()
        if lab in SKIP_LABELS:
            continue
        pos += 1
        rows.append({
            **info,
            "sentence": f"list{info['list_id']}_t{info['trial_id']}",
            "position_in_trial": pos,
            "phoneme": lab,
            "onset": float(iv.minTime),
            "offset": float(iv.maxTime),
            "duration_ms": (iv.maxTime - iv.minTime) * 1000,
            "wav_path": str(wav),
        })
    return rows


def build_tokens(corpus_root, out_path):
    print("Parsing TextGrids...")
    meta = load_metadata(corpus_root / "2/metadata_RUFR.csv")
    tg_root = corpus_root / "2/wav_et_textgrids/FRcorp_textgrids_only"

    rows = []
    for tg_path in glob.glob(str(tg_root / "**/*.TextGrid"), recursive=True):
        info = parse_filename(Path(tg_path).stem)
        if info:
            rows.extend(extract_tokens(tg_path, info))

    tokens = (
        pd.DataFrame(rows)
        .merge(meta, on="speaker", how="left")
        .sort_values(["speaker", "phoneme", "list_id", "trial_id", "onset"])
        .reset_index(drop=True)
    )
    tokens["repetition"] = tokens.groupby(["speaker", "phoneme"]).cumcount() + 1
    tokens.to_csv(out_path, index=False)

    print(f"Built tokens.csv: {len(tokens)} rows, {tokens['speaker'].nunique()} speakers")
    return tokens


tokens = build_tokens(CORPUS_ROOT, OUT / "tokens.csv")
tokens.head(3)

Parsing TextGrids...
Built tokens.csv: 22919 rows, 19 speakers


,speaker,l1,list_id,trial_id,sentence,position_in_trial,phoneme,onset,offset,duration_ms,wav_path,gender,repetition
0,AB,L1_RU,1,1,list1_t1,10,a,2.80000,2.97,170.00,/kaggle/input/datasets/alenamuravyeva/ru-fr-in...,F,1
1,AB,L1_RU,1,1,list1_t1,13,a,3.23366,3.39,156.34,/kaggle/input/datasets/alenamuravyeva/ru-fr-in...,F,2
2,AB,L1_RU,1,2,list1_t2,11,a,2.60000,2.75,150.00,/kaggle/input/datasets/alenamuravyeva/ru-fr-in...,F,3


## 3. Whisper & XlS-R feature extraction

In [5]:
import gc
import numpy as np
import torch
import librosa
from tqdm.auto import tqdm
from transformers import (
    WhisperModel, WhisperFeatureExtractor,
    Wav2Vec2Model, Wav2Vec2FeatureExtractor,
)


WHISPER_BACKEND = {
    "model_cls": WhisperModel,
    "fe_cls": WhisperFeatureExtractor,
    "input_key": "input_features",
    "pad_seconds": 30,
    "encode": lambda model, x: model.encoder(x, output_hidden_states=True).hidden_states,
    "hidden_dim": lambda model: model.config.d_model,
}

XLSR_BACKEND = {
    "model_cls": Wav2Vec2Model,
    "fe_cls": Wav2Vec2FeatureExtractor,
    "input_key": "input_values",
    "pad_seconds": None,
    "encode": lambda model, x: model(x, output_hidden_states=True).hidden_states,
    "hidden_dim": lambda model: model.config.hidden_size,
}


def time_to_frame(t, n_frames, audio_dur):
    return int(np.clip(t / audio_dur * n_frames, 0, n_frames))


def slice_pool(hs, on, off, audio_dur):
    n = hs.shape[0]
    f0 = time_to_frame(on, n, audio_dur)
    f1 = max(f0 + 1, time_to_frame(off, n, audio_dur))
    return hs[f0:f1].mean(0)


@torch.no_grad()
def encode_file(wav, fe, model, layers, backend, sr, device):
    if backend["pad_seconds"]:
        ctx = backend["pad_seconds"] * sr
        wav = np.pad(wav, (0, max(0, ctx - len(wav))))[:ctx]
    inputs = fe(wav, sampling_rate=sr, return_tensors="pt")
    x = inputs[backend["input_key"]].to(device)
    h = backend["encode"](model, x)
    return [h[l].squeeze(0).float().cpu().numpy() for l in layers], len(wav) / sr


def extract_features(tokens, model_id, layers, sr, device, out_path, backend, desc):
    fe = backend["fe_cls"].from_pretrained(model_id)
    model = backend["model_cls"].from_pretrained(model_id, output_hidden_states=True).to(device).eval()
    hidden_dim = backend["hidden_dim"](model)

    vecs = {l: np.zeros((len(tokens), hidden_dim), dtype=np.float32) for l in layers}

    for wav_path, sub in tqdm(tokens.groupby("wav_path", sort=False), desc=desc):
        try:
            wav, _ = librosa.load(wav_path, sr=sr)
        except Exception as e:
            print(f"FAIL {wav_path}: {e}")
            continue

        hs_list, audio_dur = encode_file(wav, fe, model, layers, backend, sr, device)
        idx = sub.index.to_numpy()
        ons = sub["onset"].to_numpy()
        offs = sub["offset"].to_numpy()

        for l, hs in zip(layers, hs_list):
            for k, i in enumerate(idx):
                vecs[l][i] = slice_pool(hs, ons[k], offs[k], audio_dur)

    np.savez(
        out_path,
        token_idx=np.arange(len(tokens)),
        **{f"layer_{l}": vecs[l] for l in layers},
    )
    print(f"Saved {out_path}: {vecs[layers[0]].shape}")

    del model, fe
    gc.collect()
    torch.cuda.empty_cache()


extract_features(
    tokens=tokens, model_id=WHISPER_MODEL, layers=WHISPER_LAYERS,
    sr=SR, device=device, out_path=OUT / "features_whisper.npz",
    backend=WHISPER_BACKEND, desc="Whisper",
)

extract_features(
    tokens=tokens, model_id=XLSR_MODEL, layers=XLSR_LAYERS,
    sr=SR, device=device, out_path=OUT / "features_xlsr.npz",
    backend=XLSR_BACKEND, desc="XLS-R",
)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Whisper:   0%|          | 0/1482 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLS-R:   0%|          | 0/1482 [00:00<?, ?it/s]

Saved /kaggle/working/features_xlsr.npz: (22919, 1024)


In [6]:
# verification
for f in ["features_whisper.npz", "features_xlsr.npz"]:
    p = OUT / f
    if p.exists():
        z = np.load(p)
        print(f"{f}: keys={list(z.keys())}, shapes={[z[k].shape for k in z.keys()]}, size={p.stat().st_size/1e6:.1f} MB")

features_whisper.npz: keys=['token_idx', 'layer_4', 'layer_20'], shapes=[(22919,), (22919, 1024), (22919, 1024)], size=187.9 MB
features_xlsr.npz: keys=['token_idx', 'layer_4', 'layer_12', 'layer_20'], shapes=[(22919,), (22919, 1024), (22919, 1024), (22919, 1024)], size=281.8 MB
